# 03 · Modelado (OE2)

Fase 4 de CRISP-DM. Entrena los cinco modelos base del anteproyecto con tres estrategias frente al desbalance (sin balanceo, SMOTE-NC y ponderación de clases), en los tres momentos de predicción y con dos conjuntos de variables. Todo se evalúa con validación cruzada estratificada de 5 pliegues **sobre el entrenamiento**; el conjunto de prueba no se usa en este cuaderno.

**Tiempo estimado en Colab: entre 10 y 20 minutos.** No cierres la pestaña mientras corre.

In [ ]:
# Preparar el entorno (igual que en el cuaderno 00)
import os, sys
REPO_URL = "https://github.com/andresjaimes2004/tesis-desercion.git"
REPO_DIR = "/content/tesis-desercion"
if "google.colab" in sys.modules and not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Unknown solver options")  # aviso interno de SciPy

import pandas as pd
from src import config, plots, prep, models
from src.data import load_raw, binarize_target
from src.utils import set_seed

set_seed()
pd.set_option("display.max_columns", 50)
df = prep.add_no_enrollment_flags(binarize_target(load_raw()))
idx_train, idx_test = prep.load_split()      # la misma partición de la Fase 3
print("Entrenamiento:", len(idx_train), "| Prueba (no se usa aquí):", len(idx_test))

## 1. Validación cruzada de los modelos base

5 modelos × 3 estrategias × 3 momentos × 2 conjuntos de variables = 90 combinaciones, cada una con 5 pliegues. Cada línea impresa es una combinación terminada.

In [ ]:
conjuntos = {"completo": (), "sin no significativas": prep.non_significant()}
pliegues = models.cv_base_models(df, idx_train, feature_sets=conjuntos)
pliegues.to_csv(config.REPORTS / "modelos_base_pliegues.csv", index=False, encoding="utf-8-sig")
resumen = models.summarize(pliegues)
resumen.to_csv(config.REPORTS / "modelos_base_resumen.csv", index=False, encoding="utf-8-sig")

## 2. Matriz comparativa (cierre del 2.º semestre, conjunto completo)

In [ ]:
cols = ["modelo", "estrategia"] + [f"{m}_media" for m in models.METRICS]
resumen.query("momento == 'semestre_2' and conjunto == 'completo'")[cols].round(3)

## 3. ¿Conjunto completo o reducido?

Criterio de parsimonia fijado antes de ver resultados: si el AUC-ROC del conjunto sin las 7 variables no significativas no queda más de un error estándar por debajo del completo, se prefiere el reducido por ser más simple.

In [ ]:
comparacion_conjuntos = models.feature_set_comparison(resumen)
comparacion_conjuntos.to_csv(config.REPORTS / "comparacion_conjuntos.csv", index=False, encoding="utf-8-sig")
comparacion_conjuntos.round(4)

## 4. Modelos con mayor potencial

Mejor estrategia de cada modelo según el AUC-ROC, que no depende del umbral de decisión (el umbral se ajusta en la Fase 5).

In [ ]:
ranking = models.ranking(resumen)
ranking.to_csv(config.REPORTS / "ranking_modelos.csv", index=False, encoding="utf-8-sig")
ranking[cols].round(3)

## 5. Figuras

In [ ]:
plots.strategy_comparison(resumen)
plots.moments_comparison(resumen)

## 6. Resumen para el documento

Ejecutar esta celda y enviar su salida completa.

In [ ]:
pd.set_option("display.width", 250)
m = ["modelo", "estrategia", "exactitud_media", "sensibilidad_media", "sensibilidad_de", "precision_media",
     "f1_media", "auc_roc_media", "auc_roc_de", "brier_media"]
for mom in config.MOMENTS:
    print(f"\nMATRIZ {mom.upper()} (conjunto completo):")
    print(resumen.query("momento == @mom and conjunto == 'completo'")[m].round(3).to_string(index=False))
print("\nCOMPLETO VS. REDUCIDO (AUC-ROC):")
print(comparacion_conjuntos.round(4).to_string(index=False))
print("\nRANKING SEMESTRE 2:")
print(ranking[m].round(3).to_string(index=False))

## 7. Guardar resultados en GitHub

La siguiente celda empaqueta la carpeta `reports/` en `reports_fase4.zip`. Descárgalo desde el panel de archivos, descomprímelo y sube su contenido a la carpeta `reports` del repositorio con el mensaje `Fase 4: modelado`. Luego guarda este cuaderno con Archivo > Guardar una copia en GitHub.

In [ ]:
!cd /content/tesis-desercion && zip -qr /content/reports_fase4.zip reports && echo "Listo: /content/reports_fase4.zip"